# MP14 — Neural Network from Scratch

Expanded executable lesson. Prerequisites: preceding mathematics lessons. Derive the calculation, check a hand example, reproduce the controlled experiment and interpret its limits. Learner mastery remains unassessed.

## Setup
Install the repository development dependencies. `python scripts/training_experiments.py` regenerates the experiment using existing checksum-pinned NAIP scenes; the notebook reads its committed record without downloading data. Arrays use sample-by-feature shape and float64.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp14_tools import linear_loss,initialize,mlp_loss,finite_difference_error
record=json.loads((root/'reports/mp13-mp15-training.json').read_text())


## Mathematics and worked example
A one-hidden-layer network maps X (N×D) to Z=XW1+b1 (N×H), then A=tanh(Z) (N×H), then P=AW2+b2 (N×O). W1 is D×H, b1 is H, W2 is H×O and b2 is O. Biases broadcast across samples, not features. Tanh is smooth, with derivative 1-A²; it saturates near ±1 and can make gradients small. Without a nonlinear activation, stacked affine layers collapse to one affine layer.

With L=mean((P-Y)²), set delta=2(P-Y)/(N*O). Then dW2=Aᵀ delta, db2=sum_samples(delta), hidden_delta=(delta W2ᵀ) elementwise (1-A²), dW1=Xᵀ hidden_delta, and db1=sum_samples(hidden_delta). This is the chain rule in matrix form. Every gradient has the same shape as its parameter.

Worked one-sample case: x=1,y=1,w1=0,b1=0,w2=2,b2=0. Hidden activation and prediction are zero, loss=1, output delta=-2. dw2=0, db2=-2, hidden delta=-4, dw1=-4 and db1=-4. Although dw2 is zero on this sample, gradients can still flow into the first layer because w2 is nonzero. Initializing all layers to zero can prevent useful learning and preserve symmetry.

The implementation computes each derivative explicitly with NumPy. Every parameter is checked against central finite differences, and parameters are restored after each perturbation. Checks use tiny float64 fixtures so a wrong transpose, missing mean divisor or wrong tanh derivative is observable.

In [ ]:
p={'w1':np.zeros((1,1)),'b1':np.zeros(1),'w2':np.array([[2.]]),'b2':np.zeros(1)}
loss,g=mlp_loss(np.array([[1.]]),np.array([[1.]]),p)
assert loss==1 and g['w1'].item()==-4 and g['b1'].item()==-4
assert g['w2'].item()==0 and g['b2'].item()==-2
assert finite_difference_error(np.array([[1.]]),np.array([[1.]]),p)<1e-8
print(loss,g)

## Executed experiment
The recorded network has one input, eight tanh hidden units and one output. It fits sin(3x) with 1000 full-batch SGD steps at .05. Even-indexed samples train; odd-indexed samples validate. This is an interpolation exercise, not independent geographic evidence. A separate real-image exercise maps red numerical values to NIR on one training scene and evaluates a different declared validation scene; compare it with a constant training-mean baseline. Neither task is super-resolution.

In [ ]:
print(record['optimizer'],record['seed'],record['dtype'])
print('Initial/final toy losses',record['toy_history'][0],record['toy_history'][-1])

## Practice
1. What shape is dW1 for D=3,H=8? 2. Why sum bias gradients over samples? 3. Does a finite-difference pass prove the model will generalize?

## Worked solutions
1. 3×8, matching W1. 2. The same bias influences every sample, so the chain rule adds their contributions; averaging is already included in delta. 3. No. It checks local derivatives, not data quality, useful inductive bias or held-out performance.

In [ ]:
p=initialize(inputs=3,hidden=8); _,g=mlp_loss(np.ones((2,3)),np.ones((2,1)),p)
assert g['w1'].shape==(3,8)
print('Recorded gradient error:',record['finite_difference_max_absolute_error'])
print(record['real_regression'])

## Evidence and interpretation
See [training curves](../../reports/mp13-mp15-training.png) and [complete record](../../reports/mp13-mp15-training.json). A decreasing training loss is an optimization observation, not proof of geographic generalization. The real red-to-NIR exercise has distinct scene splits and a training-mean baseline, but fixed numerical scaling is not physical calibration. Reflect on a case where the constant baseline could outperform the network.

## References
[PyTorch autograd tutorial](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) and [optimization tutorial](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html). Models here are compact teaching examples, not paper reproductions or final SR systems.